# Task #1 — Load the Bearing Fault Dataset

Confirms that `train.mat`, `val.mat`, and `test.mat` open correctly and match the schema documented in [`data/README.md`](../data/README.md).

Expected contents:

| File | Samples | Data variable | Label variable |
|------|---------|---------------|----------------|
| `train.mat` | 393 (131 per class) | `trainData` | `trainLabels` |
| `val.mat` | 27 (9 per class) | `valData` | `valLabels` |
| `test.mat` | 102 (34 per class) | `testData` | `testLabels` |

Each signal is a 5,000-sample vibration window sampled at 48,828 Hz, labeled `Normal`, `InnerRaceFault`, or `OuterRaceFault`.

In [1]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io

# Works whether the notebook runs from notebooks/ or from the repo root (e.g. Colab).
DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
print("Reading from:", DATA_DIR.resolve())

Reading from: /Users/chikaosolunnadozie/Desktop/projects/MathWorks/data


In [2]:
def load_split(split):
    """Load one split, returning (N, 5000) signals and a list of N label strings."""
    mat = scipy.io.loadmat(DATA_DIR / f"{split}.mat")
    X = mat[f"{split}Data"]
    y = [str(label[0]) for label in mat[f"{split}Labels"].flatten()]
    return X, y


X_train, y_train = load_split("train")
X_val, y_val = load_split("val")
X_test, y_test = load_split("test")

splits = {
    "train": (X_train, y_train),
    "val": (X_val, y_val),
    "test": (X_test, y_test),
}

In [3]:
summary = pd.DataFrame(
    [
        {
            "split": name,
            "signals": X.shape[0],
            "samples_per_signal": X.shape[1],
            "dtype": X.dtype,
            **Counter(y),
            "min": X.min(),
            "max": X.max(),
            "mean": X.mean(),
            "std": X.std(),
        }
        for name, (X, y) in splits.items()
    ]
).set_index("split")

summary

,signals,samples_per_signal,dtype,InnerRaceFault,OuterRaceFault,Normal,min,max,mean,std
split,,,,,,,,,,
train,393,5000,float64,131,131,131,-45.97387,39.31665,-0.153787,1.287054
val,27,5000,float64,9,9,9,-23.54578,23.72603,-0.158776,1.230148
test,102,5000,float64,34,34,34,-26.88561,27.49819,-0.151315,1.291981


In [4]:
CLASSES = {"Normal", "InnerRaceFault", "OuterRaceFault"}
EXPECTED_COUNTS = {"train": 393, "val": 27, "test": 102}

for name, (X, y) in splits.items():
    assert X.shape == (EXPECTED_COUNTS[name], 5000), f"{name}: unexpected shape {X.shape}"
    assert len(y) == X.shape[0], f"{name}: {len(y)} labels for {X.shape[0]} signals"
    assert set(y) == CLASSES, f"{name}: unexpected labels {set(y) - CLASSES}"
    assert len(set(Counter(y).values())) == 1, f"{name}: classes are not balanced"
    assert np.isfinite(X).all(), f"{name}: contains NaN or Inf values"

print("All splits loaded and verified.")
print(f"Total signals: {sum(X.shape[0] for X, _ in splits.values())}")

All splits loaded and verified.
Total signals: 522


In [5]:
# Peek at one signal from each class.
for cls in sorted(CLASSES):
    i = y_train.index(cls)
    print(f"{cls:>15}  row {i:>3}  first 5 values: {np.round(X_train[i, :5], 4)}")

 InnerRaceFault  row   0  first 5 values: [-0.0986  0.6682 -0.8851 -0.3451  0.1814]
         Normal  row 262  first 5 values: [ 0.5401  0.2835  0.9065  0.7474 -0.8117]
 OuterRaceFault  row 131  first 5 values: [-0.8741 -1.5575 -0.8622 -1.3693 -1.2064]


## Result

All three `.mat` files open with `scipy.io.loadmat`, are balanced across the three fault classes, contain no NaN/Inf values, and match the documented `(N x 5000)` float64 shape.